# Notebook 14 - Mixed Full+ROI Training Ablation

Builds a research-only mixed dataset view, trains a new adapter, scores full-image/ROI/hybrid inference, and pushes standardized result reports.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            os.chdir(repo_root)
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    os.chdir(CLONE_TARGET)
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
from scripts.colab_notebook_bootstrap_helpers import setup_notebook_environment
ROOT = setup_notebook_environment(install_requirements=True, print_tokens=True)
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
from scripts.colab_roi_ablation import commit_and_push_ablation_results, run_mixed_full_roi_training_ablation

ABLATION_NAME = 'mixed_full_roi_training'
DATASET_KEY = 'tomato__fruit'
CROP_NAME = 'tomato'
PART_NAME = 'fruit'
SOURCE_DATASET_ROOT = ROOT / 'data' / 'prepared_runtime_datasets'
OUTPUT_DIR = ROOT / 'docs' / 'ablation_results' / ABLATION_NAME
RUNTIME_ROOT = ROOT / 'outputs' / 'ablation_runtime' / ABLATION_NAME
CONFIG_ENV = 'colab'
DEVICE = 'cuda'

# Keep these explicit so Colab runs are controllable. Increase EPOCHS for the final expensive run.
EPOCHS = 12
BATCH_SIZE = 64
LEARNING_RATE = 6e-5
NUM_WORKERS = 8
PIN_MEMORY = True

print(f'[CONFIG] ablation={ABLATION_NAME} dataset={DATASET_KEY} output={OUTPUT_DIR}')
print(f'[CONFIG] epochs={EPOCHS} batch={BATCH_SIZE} lr={LEARNING_RATE} runtime={RUNTIME_ROOT}')


In [ ]:
if not Path(SOURCE_DATASET_ROOT / DATASET_KEY).is_dir():
    raise FileNotFoundError(f"Dataset not found: {SOURCE_DATASET_ROOT / DATASET_KEY}")

summary = run_mixed_full_roi_training_ablation(
    source_dataset_root=SOURCE_DATASET_ROOT,
    output_dir=OUTPUT_DIR,
    runtime_root=RUNTIME_ROOT,
    dataset_key=DATASET_KEY,
    crop_name=CROP_NAME,
    part_name=PART_NAME,
    config_env=CONFIG_ENV,
    device=DEVICE,
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add mixed full ROI training ablation results',
)
{'summary': summary['inference_summaries'], 'adapter_dir': summary['adapter_dir'], 'git': git_result}
